# Data Cleaning

## Objectives

- Clean the raw airline passenger satisfaction dataset.
- Remove columns that are not useful for analysis.
- Handle missing values identified during the initial data inspection.
- Check categorical and numerical values for consistency.
- Review unusual values before deciding whether any treatment is required.
- Save a cleaned version of the dataset for later analysis.

## Inputs

- `data/raw_data/train.csv`

## Outputs

- `data/clean_data/airline_clean.csv`
- A cleaned dataset ready for exploratory data analysis, statistical analysis and machine learning.



---

# Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\hanee\\OneDrive\\Desktop\\Project2\\airline_passenger_satisfaction_analyst\\airline-passenger-satisfaction\\jupyter_notebooks'

In [2]:
if os.path.basename(current_dir) == "jupyter_notebooks":
    os.chdir(os.path.dirname(current_dir))

print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\hanee\OneDrive\Desktop\Project2\airline_passenger_satisfaction_analyst\airline-passenger-satisfaction


## Load the Raw Dataset

Load the original airline passenger satisfaction dataset so that the cleaning steps are applied to a fresh copy of the raw data.

In [3]:
import pandas as pd
df = pd.read_csv("data/raw_data/train.csv")
df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


---

### Dataset Shape Before Cleaning

Check the number of rows and columns before applying any cleaning steps.

In [5]:
df.shape

(103904, 25)

## Remove Unnecessary Columns

The initial data inspection identified `Unnamed: 0` and `id` as possible identifier columns. Before removing them, check whether each column contains a unique value for every passenger record.

In [7]:
df[["Unnamed: 0", "id"]].nunique()

Unnamed: 0    103904
id            103904
dtype: int64

---

Both columns contain 103,904 unique values, equal to the total number of rows in the dataset. This confirms that they act as record identifiers and are not useful features for the analysis. They will therefore be removed.

In [9]:
df = df.drop(columns=["Unnamed: 0", "id"])

In [11]:
df.shape

(103904, 23)

## Missing Values

Check the dataset for missing values after removing the unnecessary identifier columns. The initial inspection found missing values in `Arrival Delay in Minutes`, so these will be investigated before deciding how to handle them.

In [13]:
df.isnull().sum()

Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             310
satisfaction                           0
dtype: int64

### Investigate Missing Arrival Delay Values

Inspect the records where `Arrival Delay in Minutes` is missing to understand whether there is an appropriate way to handle these values.

In [15]:
missing_arrival_delay = df[df["Arrival Delay in Minutes"].isnull()]

missing_arrival_delay[
    ["Departure Delay in Minutes", "Arrival Delay in Minutes", "satisfaction"]
].head(10)

,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
213,31,NaN,satisfied
1124,38,NaN,neutral or dissatisfied
1529,11,NaN,neutral or dissatisfied
2004,41,NaN,neutral or dissatisfied
2108,1,NaN,neutral or dissatisfied
2485,3,NaN,satisfied
2630,0,NaN,satisfied
3621,17,NaN,neutral or dissatisfied
4041,6,NaN,satisfied
4490,22,NaN,neutral or dissatisfied


### Relationship Between Departure and Arrival Delay

Since the records with missing arrival delay contain a range of departure delay values, examine the relationship between the two delay variables before deciding how to treat the missing values.

In [17]:
df[["Departure Delay in Minutes", "Arrival Delay in Minutes"]].corr()

,Departure Delay in Minutes,Arrival Delay in Minutes
Departure Delay in Minutes,1.000000,0.965481
Arrival Delay in Minutes,0.965481,1.000000


### Handle Missing Arrival Delay Values

Departure delay and arrival delay have a strong positive correlation of approximately 0.97. However, only 310 records have a missing `Arrival Delay in Minutes` value, representing approximately 0.30% of the dataset.

Rather than estimating these values and introducing potentially inaccurate data, the incomplete records will be removed.

In [21]:
df = df.dropna(subset=["Arrival Delay in Minutes"])

In [22]:
df.shape

(103594, 23)

### Validate Missing Value Treatment

Confirm that no missing values remain in the dataset after removing the incomplete arrival delay records.

In [24]:
df.isnull().sum().sum()

np.int64(0)

## Duplicate Records

Check for duplicated records again after removing the identifier columns. Records that previously had unique identifiers may otherwise contain identical analytical data.

In [26]:
df.duplicated().sum()

np.int64(0)

### Duplicate Check Result

No completely duplicated records were found after removing the identifier columns. Therefore, no duplicate records need to be removed.

## Categorical Data Consistency

Inspect the unique values in the categorical columns to identify any inconsistent or unexpected category labels before further analysis.

In [28]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"{column}:")
    print(df[column].unique())
    print()

Gender:
<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

Customer Type:
<ArrowStringArray>
['Loyal Customer', 'disloyal Customer']
Length: 2, dtype: str

Type of Travel:
<ArrowStringArray>
['Personal Travel', 'Business travel']
Length: 2, dtype: str

Class:
<ArrowStringArray>
['Eco Plus', 'Business', 'Eco']
Length: 3, dtype: str

satisfaction:
<ArrowStringArray>
['neutral or dissatisfied', 'satisfied']
Length: 2, dtype: str



C:\Users\hanee\AppData\Local\Temp\ipykernel_22964\3289315658.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


### Categorical Consistency Result

The categorical variables contain the expected category labels, with no unexpected or duplicate categories identified. Therefore, no categorical value corrections are required.

## Service Rating Validation

Check the minimum and maximum values of the service-rating variables to identify any values outside the expected 0 to 5 range.

In [30]:
service_columns = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

df[service_columns].agg(["min", "max"])

,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness
min,0,0,0,0,0,0,0,0,0,0,1,0,0,0
max,5,5,5,5,5,5,5,5,5,5,5,5,5,5


### Service Rating Validation Result

All service-rating variables contain values within the expected 0 to 5 range. No values below 0 or above 5 were identified, so no corrections are required for these variables.

## Delay Outlier Investigation

The initial data inspection identified unusually high maximum values in the departure and arrival delay variables. These values will be investigated before deciding whether any outlier treatment is appropriate.

In [32]:
df[["Departure Delay in Minutes", "Arrival Delay in Minutes"]].describe()

,Departure Delay in Minutes,Arrival Delay in Minutes
count,103594.000000,103594.000000
mean,14.747939,15.178678
std,38.116737,38.698682
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,12.000000,13.000000
max,1592.000000,1584.000000


### Inspect Extreme Delay Records

The delay variables are strongly right-skewed, with most passengers experiencing relatively small delays while a small number have very large delays. The records with the highest delays will be inspected before deciding whether outlier treatment is necessary.

In [34]:
df[
    ["Departure Delay in Minutes", "Arrival Delay in Minutes", "satisfaction"]
].sort_values(
    by="Departure Delay in Minutes",
    ascending=False
).head(10)

,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
83741,1592,1584.0,neutral or dissatisfied
6744,1305,1280.0,satisfied
61310,1017,1011.0,satisfied
72206,978,970.0,neutral or dissatisfied
80182,933,920.0,satisfied
61287,930,952.0,neutral or dissatisfied
61528,921,924.0,neutral or dissatisfied
27732,859,860.0,satisfied
51860,853,823.0,neutral or dissatisfied
37096,750,729.0,satisfied


### Delay Outlier Decision

The largest departure delays are accompanied by similarly large arrival delays, indicating that the extreme values are internally consistent rather than obvious data-entry errors.

Although the delay variables are strongly right-skewed, unusually long flight delays are plausible real-world observations. Therefore, these records will be retained rather than removed or capped. Their distribution will be considered during further analysis and modelling.

## Final Data Quality Validation

Perform final checks on the cleaned dataset to confirm its shape, missing values, duplicate records, and column structure before saving the cleaned data.

In [36]:
print(f"Dataset shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate records: {df.duplicated().sum()}")
print(f"Number of columns: {df.shape[1]}")

Dataset shape: (103594, 23)
Missing values: 0
Duplicate records: 0
Number of columns: 23


## Data Cleaning Summary

The following cleaning and validation steps were completed:

- Removed `Unnamed: 0` and `id` because both contained a unique value for every record and were not useful analytical features.
- Identified 310 missing values in `Arrival Delay in Minutes`.
- Removed the 310 records with missing arrival delay values rather than estimating them.
- Confirmed that no missing values remain in the cleaned dataset.
- Confirmed that no completely duplicated records are present after removing the identifier columns.
- Checked categorical variables and found no unexpected or inconsistent category labels.
- Validated that the service-rating variables contain values within the expected 0 to 5 range.
- Investigated unusually high departure and arrival delays and retained them because the values were internally consistent and represent plausible real-world observations.

The cleaned dataset contains **103,594 rows and 23 columns** and is ready for further analysis.

## Save Cleaned Dataset

Save the cleaned dataset to the `data/clean_data/` folder so it can be used in the exploratory data analysis, statistical analysis and machine learning stages.

In [38]:
df.to_csv("data/clean_data/airline_clean.csv", index=False)

In [39]:
clean_df = pd.read_csv("data/clean_data/airline_clean.csv")

clean_df.shape

(103594, 23)

## Conclusions and Next Steps

The raw airline passenger satisfaction dataset was cleaned and validated.

The `Unnamed: 0` and `id` identifier columns were removed, 310 records with missing `Arrival Delay in Minutes` values were removed, and the remaining dataset was checked for duplicate records, inconsistent categorical values, invalid service ratings and extreme delay values.

The extreme delay values were retained because they were internally consistent and represent plausible real-world observations.

The cleaned dataset contains **103,594 rows and 23 columns** with no missing values or duplicate records.

The next step is to carry out exploratory data analysis to investigate passenger characteristics, travel patterns, service ratings and their relationships with passenger satisfaction.

---